In [14]:
''''
import pandas as pd
import geopandas as gpd
import numpy as np
from skimage.filters import threshold_otsu
import rasterio
from rasterio.merge import merge
from rasterio.features import shapes
import os
from shapely.geometry import box
'''
def create_census_geojson(input_csv,input_shp):
    import pandas as pd
    import geopandas as gpd
    import os
    canada_csv = input_csv
    shapefile_path = input_shp

    df = pd.read_csv(canada_csv, encoding='latin1')
    df = df.apply(pd.to_numeric, errors='coerce')
    print(df.head())
    # Drop columns from a pandas DataFrame
    df = df.drop(columns=['pop16', 'pop%delt', 'dwelltot', 'privd'])  
    #print(df.head())

    df = pd.read_csv(canada_csv, encoding='latin1')
    # Drop columns from a pandas DataFrame
    df = df.drop(columns=['pop16', 'pop%delt', 'dwelltot', 'privd']) 
    print(df.head())

    gdf = gpd.read_file(shapefile_path)
    gdf = gdf.to_crs(epsg=4326) # Ensure both are in the same CRS

    # merging the csv data with the shapefile data
    merged = gdf.merge(df, left_on='CTUID', right_on='CTUID')
    # Export to GeoJSON as bound_
    geojson_path = os.path.join("data/", "censustracts.geojson")
    merged.to_file(geojson_path, driver="GeoJSON")

    #example usage
    #create_census_geojson("raw-data/montreal/9810001402-eng_clean.csv","raw-data/census tracts/lct_000b21a_e.shp")


        CTUID   pop21   pop16  pop%delt  dwelltot   privd  a_land  popdens
0  9320001.00  3100.0  3101.0       0.0     723.0   660.0   26.37    117.5
1  9320002.00  1226.0  1163.0       5.4     441.0   423.0    8.80    139.4
2  9320003.00  7963.0  7248.0       9.9    2790.0  2643.0    2.17   3674.3
3  9320004.00  7591.0  7061.0       7.5    2373.0  2216.0    1.90   4004.5
4  9320005.01  4019.0  3715.0       8.2    1557.0  1459.0    1.26   3179.1
        CTUID   pop21  a_land  popdens
0  9320001.00  3100.0   26.37    117.5
1  9320002.00  1226.0    8.80    139.4
2  9320003.00  7963.0    2.17   3674.3
3  9320004.00  7591.0    1.90   4004.5
4  9320005.01  4019.0    1.26   3179.1


jupyter nbconvert --to script your_notebook.ipynb in order to convert this markdown file to python script for calls

In [ ]:
'''# Path to your shapefile
shapefile_path = "raw-data/census tracts/lct_000b21a_e.shp"

# Read the shapefile using geopandas
gdf = gpd.read_file(shapefile_path)

#convert to WGS84
gdf = gdf.to_crs(epsg=4326) 

# Export to GeoJSON
geojson_path = "raw-data/montreal/lct_000b21a_e.geojson"
gdf.to_file(geojson_path, driver="GeoJSON")
'''
# --- IGNORE --- we dont really need to run this part, just for reference

In [17]:
'''
#this block will create and export a new geojson with the csv data merged in
# Read CSV and shapefile
csv_path = "raw-data/montreal/9810001402-eng_clean.csv"
shapefile_path = "raw-data/census tracts/lct_000b21a_e.shp"

df = pd.read_csv(csv_path, encoding='latin1')
# Drop columns from a pandas DataFrame
df = df.drop(columns=['pop16', 'pop%delt', 'dwelltot', 'privd']) 
print(df.head())

gdf = gpd.read_file(shapefile_path)
gdf = gdf.to_crs(epsg=4326) # Ensure both are in the same CRS

# merging the csv data with the shapefile data
merged = gdf.merge(df, left_on='CTUID', right_on='CTUID')
# Export to GeoJSON as bound_
geojson_path = os.path.join("data/", "censustracts.geojson")
merged.to_file(geojson_path, driver="GeoJSON")'''

        CTUID    pop21  a_land  popdens
0  9320001.00  3100.00   26.37   117.50
1  9320002.00  1226.00    8.80   139.40
2  9320003.00  7963.00    2.17  3674.30
3  9320004.00  7591.00    1.90  4004.50
4  9320005.01  4019.00    1.26  3179.10


so this section is dedicated to exclusively raster operators 
i want to show and ndvi (NIR-RED/NIR+RED wavelength) index with a threshold that will only display 'healthy vegetation' the datasets for this have been picked during peak growth cycles because winter obviously doesnt have much vegetation
landsat 2 collection 2 is atmospherically compensated but needs a scaling corrective factor of (band*0.0000275-0.2) 
i will used bands 4 for red and 5 for near infra-red

In [1]:
#scaling function for landsat 2 collection 2
# Path to your input .tiff file
#input_tiff = "data/landsat/your_image.tiff" # this is just an example path
#output_tiff = "data/landsat/scaled_image.tiff" #the output path

def scale_tiff(input_tiff, prefix="scaled_"):
    
    import rasterio
    from rasterio.merge import merge
    from rasterio.features import shapes
    import os
     # Get directory and filename
    dir_name, file_name = os.path.split(input_tiff)
    output_tiff = os.path.join(dir_name, prefix + file_name)

    with rasterio.open(input_tiff) as src:
        profile = src.profile
        data = src.read(1)  # Read the first band

        # Apply scaling: (DN * 0.0000275) - 0.2
        scaled_data = (data * 0.0000275) - 0.2

        # Update profile to float32 for scaled values
        profile.update(dtype=rasterio.float32)

        # Write the scaled data to a new .tiff file
        with rasterio.open(output_tiff, 'w', **profile) as dst:
            dst.write(scaled_data.astype(rasterio.float32), 1)

    print("Scaling complete. Output saved to:", output_tiff)

# montreal usage:

#scale_tiff("raw-data/montreal/landsat2_c2/LC08_L2SP_014028_20240701_20240711_02_T1_SR_B4.TIF")
#band 4
#scale_tiff("raw-data/montreal/landsat2_c2/LC08_L2SP_014028_20240701_20240711_02_T1_SR_B5.TIF")
#band 5


Scaling complete. Output saved to: raw-data/montreal/landsat2_c2\scaled_LC08_L2SP_014028_20240701_20240711_02_T1_SR_B4.TIF
Scaling complete. Output saved to: raw-data/montreal/landsat2_c2\scaled_LC08_L2SP_014028_20240701_20240711_02_T1_SR_B5.TIF


In [4]:
# make ndvi function


def calculate_ndvi(red_band_path, nir_band_path, city):
    import numpy as np
    import rasterio
    from rasterio.merge import merge
    from rasterio.features import shapes
    import os
   # Get directory and filename
    dir_name, file_name = os.path.split(red_band_path)
    output_tiff = os.path.join(dir_name, "ndvi_" + city)
    # Open red band
    with rasterio.open(red_band_path) as red_src:
        red = red_src.read(1).astype('float32')
        profile = red_src.profile

    # Open NIR band
    with rasterio.open(nir_band_path) as nir_src:
        nir = nir_src.read(1).astype('float32')

    # NDVI calculation: (NIR - RED) / (NIR + RED)
    ndvi = (nir - red) / (nir + red)
    ndvi = np.clip(ndvi, -1, 1)  # Optional: clip values to valid NDVI range

    # Update profile for output
    profile.update(dtype=rasterio.float32, count=1)

    # Write NDVI to new tiff
    with rasterio.open(output_tiff, 'w', **profile) as dst:
        dst.write(ndvi, 1)

    print("NDVI calculation complete. Output saved to:", output_tiff)

# Example usage for montreal:
#calculate_ndvi("raw-data/montreal/landsat2_c2/LC08_L2SP_014028_20240701_20240711_02_T1_SR_B4.TIF", "raw-data/montreal/landsat2_c2/LC08_L2SP_014028_20240701_20240711_02_T1_SR_B5.TIF",city="montreal.tiff")

C:\Users\BlackBox\AppData\Local\Temp\ipykernel_39280\156452847.py:23: RuntimeWarning: invalid value encountered in divide
  ndvi = (nir - red) / (nir + red)


NDVI calculation complete. Output saved to: raw-data/montreal/landsat2_c2\ndvi_montreal.tiff


we have an ndvi map now. we need to find a threshold for it. 
i have found that the otsu method might be best for python automation

In [7]:
def otsu_ndvi_threshold(ndvi_tiff):
    import numpy as np
    from skimage.filters import threshold_otsu
    import rasterio
    from rasterio.merge import merge
    from rasterio.features import shapes
    import os
    # Read NDVI raster
    with rasterio.open(ndvi_tiff) as src:
        ndvi = src.read(1)
        profile = src.profile

    # Flatten and remove NaN values for threshold calculation
    ndvi_flat = ndvi.flatten()
    ndvi_flat = ndvi_flat[~np.isnan(ndvi_flat)]

    # Compute Otsu threshold
    thresh = threshold_otsu(ndvi_flat)
    print(f"Otsu threshold for NDVI: {thresh}")

    # Create binary mask: 1 for healthy vegetation, 0 otherwise
    mask = (ndvi >= thresh).astype(np.uint8)

    # Save mask as new tiff
    profile.update(dtype=rasterio.uint8, count=1)
    mask_tiff = ndvi_tiff.replace('.tiff', '_mask.tiff')
    with rasterio.open(mask_tiff, 'w', **profile) as dst:
        dst.write(mask, 1)

    print("Mask saved to:", mask_tiff)
    return thresh, mask_tiff

# Example usage montreal:
#otsu_ndvi_threshold("raw-data/montreal/landsat2_c2/ndvi_montreal.tiff")

Otsu threshold for NDVI: 0.2860080897808075
Mask saved to: raw-data/montreal/landsat2_c2/ndvi_montreal_mask.tiff
(np.float32(0.2860081), 'raw-data/montreal/landsat2_c2/ndvi_montreal_mask.tiff')


In [31]:
# turn the mask into a geojson since it will show the healthy vegetation areas based on the threhold value
def mask_to_geojson(mask_tiff):
    import pandas as pd
    import geopandas as gpd
    import numpy as np
    from skimage.filters import threshold_otsu
    import rasterio
    from rasterio.merge import merge
    from rasterio.features import shapes
    import os
    from shapely.geometry import box
    # Get directory and filename
    dir_name, file_name = os.path.split(mask_tiff)
    # Remove ".tiff" or ".tif" from filename
    base_name = file_name.replace('.tiff', '').replace('.tif', '')
    output_geojson = os.path.join(dir_name, f"greenspace_{base_name}.geojson")

    with rasterio.open(mask_tiff) as src:
        mask = src.read(1)
        mask = mask.astype('uint8')
        transform = src.transform
        crs = src.crs

        # Extract shapes (polygons) where mask == 1
        results = (
            {"properties": {"value": v}, "geometry": s}
            for s, v in shapes(mask, mask=mask==1, transform=transform)
            if v == 1
        )

        gdf = gpd.GeoDataFrame.from_features(list(results))
        gdf = gdf.set_crs(crs)
        gdf = gdf.to_crs(epsg=4326) # Ensure both are in the same CRS

        # Example: save simplified GeoJSON
        gdf.to_file("raw-data/montreal/output_simplified.geojson", driver="GeoJSON")
        print(f"GeoJSON saved to: {output_geojson}")

# Example usage montreal:
#mask_to_geojson("raw-data/montreal/landsat2_c2/ndvi_montreal_mask.tiff")

GeoJSON saved to: raw-data/montreal/landsat2_c2\greenspace_ndvi_montreal_mask.geojson


in theory now, for other cities i can just call the functions and avoid a lot of code repetitions. 

In [40]:
# code to cut geographic extent of a geojson to a specific bounding box know that the bounds should be the same as the map bounds in webapp.js
#since we are pre processing and cities are different sizes, this needs to be  manually determined and entered for each geojson city

# Load greenspace polygons
#greenspace_gdf = gpd.read_file("raw-data/montreal/landsat2_c2/output_simplified.geojson")

# Define your bounding box (minx, miny, maxx, maxy)
#minx, miny = -74.00, 45.35
#maxx, maxy = -73.40, 45.75
#bbox = box(minx, miny, maxx, maxy)

# Clip greenspace polygons to bounding box
#clipped_gdf = greenspace_gdf.clip(bbox)

# Save clipped GeoJSON
#clipped_gdf.to_file("data/montreal/output_simplified_clipped.geojson", driver="GeoJSON")


def clip_geojson_to_bbox(input_file):
    import geopandas as gpd
    from shapely.geometry import box

    # Prompt user for bounding box coordinates
    print("Enter the bounding box coordinates:")
    minx = float(input("Minimum longitude (minx): "))
    miny = float(input("Minimum latitude (miny): "))
    maxx = float(input("Maximum longitude (maxx): "))
    maxy = float(input("Maximum latitude (maxy): "))
    
    # Load the GeoJSON file
    try:
        greenspace_gdf = gpd.read_file(input_file)
    except Exception as e:
        print(f"Error reading the file: {e}")
        return
    
    # Define the bounding box
    bbox = box(minx, miny, maxx, maxy)
    
    # Clip the GeoJSON to the bounding box
    try:
        clipped_gdf = greenspace_gdf.clip(bbox)
    except Exception as e:
        print(f"Error clipping the GeoJSON: {e}")
        return
    
    # Save the clipped GeoJSON
    output_file = input_file.replace(".geojson", "_clipped.geojson")
    try:
        clipped_gdf.to_file(output_file, driver="GeoJSON")
        print(f"Clipped GeoJSON saved to: {output_file}")
    except Exception as e:
        print(f"Error saving the clipped GeoJSON: {e}")

# Call example the function
#clip_geojson_to_bbox("raw-data/montreal/output_simplified.geojson")


Enter the bounding box coordinates:
Clipped GeoJSON saved to: raw-data/montreal/output_simplified_clipped.geojson


In [34]:
#should be a deprecated block now since we made the function above
#  ----------- for this to work, we must find how much greenspace area is within each census tract
# the we create a column where we divide the population by the greenspace area to get a per capita value
# later we can use all that data to generate a rating for how green the city is
# Load greenspace polygons (GeoJSON)
#
#
#
# im hoping to skip this by forcing census tracts to display on top. reduces data size this way
# we still should make this file for the statistical math to be easier

#greenspace_gdf = gpd.read_file("data/montreal/output_simplified_clipped.geojson")

# Load census tracts (GeoJSON or Shapefile)
#census_gdf = gpd.read_file("data/censustracts.geojson")

# Ensure both are in the same CRS
#greenspace_gdf = greenspace_gdf.to_crs(epsg=4326)
#census_gdf = census_gdf.to_crs(epsg=4326)

# Spatial join: append census tract attributes to greenspace polygons
#joined = gpd.sjoin(greenspace_gdf, census_gdf, how="left", predicate="intersects")

# Save the result
#joined.to_file("raw-data/montreal/greenspace_with_census.geojson", driver="GeoJSON")

#print("Processing complete. file  saved to data/montreal/greenspace_with_census.geojson")



# im hoping to skip this by forcing census tracts to display on top. reduces data size this way
    # we still should make this file for the statistical math to be easier
# Save the clipped GeoJSON in the same directory as the input file
def combine_greenspace_with_census(input_file):
    import geopandas as gpd
    import os
    # Get directory of input file
    input_dir = os.path.dirname(input_file)
    clipped_file = os.path.join(input_dir, os.path.basename(input_file))

    try:
        # Load the clipped GeoJSON file
        clipped_gdf = gpd.read_file(clipped_file)
        print(f"Clipped GeoJSON loaded from: {clipped_file}")
    except Exception as e:
        print(f"Error reading the clipped GeoJSON file: {e}")
        return

    try:
        # Load the census GeoJSON file
        census_gdf = gpd.read_file("data/censustracts.geojson")
    except Exception as e:
        print(f"Error reading the censustracts file: {e}")
        return

    # Ensure both GeoDataFrames are in the same CRS
    clipped_gdf = clipped_gdf.to_crs(epsg=4326)
    census_gdf = census_gdf.to_crs(epsg=4326)

    try:
        # Spatial join: append census tract attributes to greenspace polygons
        joined = gpd.sjoin(clipped_gdf, census_gdf, how="left", predicate="intersects")
    except Exception as e:
        print(f"Error performing spatial join: {e}")
        return

    # Save the combined GeoJSON in the same directory as the input file
    combined_file = os.path.join(input_dir, os.path.basename(input_file).replace(".geojson", "_with_census.geojson"))
    try:
        joined.to_file(combined_file, driver="GeoJSON")
        print(f"Combined GeoJSON saved to: {combined_file}")
    except Exception as e:
        print(f"Error saving the combined GeoJSON: {e}")
        return

#combine_greenspace_with_census("raw-data/montreal/output_simplified_clipped.geojson")



Clipped GeoJSON loaded from: raw-data/montreal\output_simplified_clipped.geojson
Combined GeoJSON saved to: raw-data/montreal\output_simplified_clipped_with_census.geojson


we will now create a pipeline to produce a statistical representation of the census tracts
this will also be used to create summary stats

In [ ]:
'''# Load census tracts (GeoJSON or Shapefile)
greenspace_capita = gpd.read_file("raw-data/montreal/output_simplified_clipped_with_census.geojson")
greenspace_capita = greenspace_capita.to_crs(epsg=32188) # Ensure  CRS

# Calculate greenspace area per feature (in square meters)
greenspace_capita['greenspace_area'] = greenspace_capita.geometry.area
greenspace_capita['LANDAREA']= greenspace_capita['LANDAREA']*1e6

# Ensure 'CTUID' and 'pop21' are valid
greenspace_capita['CTUID'] = greenspace_capita['CTUID'].astype(str)
greenspace_capita['pop21'] = pd.to_numeric(greenspace_capita['pop21'], errors='coerce')
greenspace_capita['LANDAREA'] = pd.to_numeric(greenspace_capita['LANDAREA'], errors='coerce')

# Consolidate greenspace area by CTUID (sum all polygons in each tract)
greenspace_sum = greenspace_capita.groupby('CTUID').agg({
    'greenspace_area': 'sum',
    'pop21': 'first',
    'LANDAREA': 'first',
    'geometry': 'first'
}).reset_index()

# Calculate greenspace per tract (area/land area)m
greenspace_sum['greenspace_per_tract'] = greenspace_sum['greenspace_area'] / greenspace_sum['LANDAREA']

# Calculate greenspace per capita (area/population) m/person
greenspace_sum['greenspace_per_capita'] = greenspace_sum['greenspace_area'] / greenspace_sum['pop21']

# Save consolidated GeoJSON
greenspace_sum = gpd.GeoDataFrame(greenspace_sum, geometry='geometry', crs='EPSG:32188')
greenspace_sum = greenspace_sum.to_crs(epsg=4326)
greenspace_sum.to_file("raw-data/montreal/greenspace_capita.geojson", driver="GeoJSON")
# do math, cut the polygons, check the percentage of greenspace per census tract,use that percentage to get area of greenspace, 
# then divide population by greenspace area to get per capita value, then use that to generate a rating for how green the city is
# the file above should already have the census data merged and the greenspace polygons cut to the census tracts
#remove the census tract data from the greenspace polygons to reduce file size
greenspace_final = gpd.read_file("raw-data/montreal/greenspace_capita.geojson")
greenspace_final = greenspace_final.drop(columns=['CTNAME', 'PRUID', 'CDUID', 'CDNAME', 'CSDUID', 'CSDNAME', 'CMAUID', 'CMANAME', 'CDTYPE', 'CSDTYPE', 'TYPE', 'pop21', 'LANDAREA'])
greenspace_final.to_file("data/montreal/greenspace_final.geojson", driver="GeoJSON")

'''


def process_greenspace_census_data(input_file, output_dir="data"):
    import geopandas as gpd
    import pandas as pd
    import os
    """
    Processes greenspace census data to calculate greenspace per tract and per capita.

    Parameters:
        input_file (str): Path to the input GeoJSON file containing greenspace and census data.
        output_dir (str): Directory where the output files will be saved. Default is "data".

    Outputs:
        - greenspace_capita.geojson: GeoJSON with greenspace per tract and per capita values.
        - greenspace_final.geojson: GeoJSON with reduced columns for final use.
    """
    try:
        # Load greenspace census data
        greenspace_capita = gpd.read_file(input_file, output_dir)
        greenspace_capita = greenspace_capita.to_crs(epsg=32188)  # Ensure CRS is in meters for area calculations

        # Calculate greenspace area per feature (in square meters)
        greenspace_capita['greenspace_area'] = greenspace_capita.geometry.area
        greenspace_capita['LANDAREA'] = greenspace_capita['LANDAREA'] * 1e6  # Convert LANDAREA to square meters

        # Ensure 'CTUID' and 'pop21' are valid
        greenspace_capita['CTUID'] = greenspace_capita['CTUID'].astype(str)
        greenspace_capita['pop21'] = pd.to_numeric(greenspace_capita['pop21'], errors='coerce')
        greenspace_capita['LANDAREA'] = pd.to_numeric(greenspace_capita['LANDAREA'], errors='coerce')

        # Consolidate greenspace area by CTUID (sum all polygons in each tract)
        greenspace_sum = greenspace_capita.groupby('CTUID').agg({
            'greenspace_area': 'sum',
            'pop21': 'first',
            'LANDAREA': 'first',
            'geometry': 'first'
        }).reset_index()

        # Calculate greenspace per tract (area/land area) in m²
        greenspace_sum['greenspace_per_tract'] = greenspace_sum['greenspace_area'] / greenspace_sum['LANDAREA']

        # Calculate greenspace per capita (area/population) in m²/person
        greenspace_sum['greenspace_per_capita'] = greenspace_sum['greenspace_area'] / greenspace_sum['pop21']

        # Save consolidated GeoJSON
        greenspace_sum = gpd.GeoDataFrame(greenspace_sum, geometry='geometry', crs='EPSG:32188')
        greenspace_sum = greenspace_sum.to_crs(epsg=4326)  # Convert back to WGS84 for GeoJSON
        input_dir = os.path.dirname(input_file)
        if not input_dir:
            input_dir = output_dir
        greenspace_capita_path = os.path.join(
            input_dir,
            os.path.basename(input_file).replace(".geojson", "_greenspace_capita.geojson")
        )
        greenspace_sum.to_file(greenspace_capita_path, driver="GeoJSON")
        print(f"Greenspace capita data saved to: {greenspace_capita_path}")

        # Remove unnecessary columns to reduce file size
        greenspace_final = greenspace_sum.drop(columns=[
            'CTNAME', 'PRUID', 'CDUID', 'CDNAME', 'CSDUID', 'CSDNAME',
            'CMAUID', 'CMANAME', 'CDTYPE', 'CSDTYPE', 'TYPE', 'pop21', 'LANDAREA'
        ], errors='ignore')  # Use `errors='ignore'` to avoid issues if columns are missing
        greenspace_final_path = f"{output_dir}/greenspace_final.geojson"
        greenspace_final.to_file(greenspace_final_path, driver="GeoJSON")
        print(f"Final greenspace data saved to: {greenspace_final_path}")

    except Exception as e:
        print(f"An error occurred: {e}")
        print(f"we are at processing greenspace with census data")

# Example usage
#process_greenspace_census_data("raw-data/montreal/output_simplified_clipped_with_census.geojson", output_dir="data/montreal")




Greenspace capita data saved to: data/montreal\greenspace_capita.geojson
Final greenspace data saved to: data/montreal\greenspace_final.geojson


In [ ]:
#we use this function now
def process_greenspace_data(input_file, output_dir):
    """
    Processes greenspace data to calculate greenspace per tract and per capita.

    Parameters:
        input_file (str): Path to the input GeoJSON file containing greenspace and census data.
        output_dir (str): Directory where the output files will be saved. Default is "data".

    Outputs:
        - greenspace_capita.geojson: GeoJSON with greenspace per tract and per capita values.
        - greenspace_final.geojson: GeoJSON with reduced columns for final use.
    """
    import geopandas as gpd
    import pandas as pd
    import os
    import numpy as np
    print(f"we are at processing greenspace data below")
    try:
        # Load greenspace census data
        greenspace_capita = gpd.read_file(input_file)
        greenspace_capita = greenspace_capita.to_crs(epsg=32188)  # Ensure CRS is in meters for area calculations

        # Calculate greenspace area per feature (in square meters)
        greenspace_capita['greenspace_area'] = greenspace_capita.geometry.area
        greenspace_capita['LANDAREA'] = greenspace_capita['LANDAREA'] * 1e6  # Convert LANDAREA to square meters

        # Ensure 'CTUID' and 'pop21' are valid
        greenspace_capita['CTUID'] = greenspace_capita['CTUID'].astype(str)
        greenspace_capita['pop21'] = pd.to_numeric(greenspace_capita['pop21'], errors='coerce')
        greenspace_capita['LANDAREA'] = pd.to_numeric(greenspace_capita['LANDAREA'], errors='coerce')

        # Consolidate greenspace area by CTUID (sum all polygons in each tract)
        greenspace_sum = greenspace_capita.groupby('CTUID').agg({
            'greenspace_area': 'sum',
            'pop21': 'first',
            'LANDAREA': 'first',
            'geometry': 'first'
        }).reset_index()

        # Calculate greenspace per capita (area/population) in m²/person
        greenspace_sum['greenspace_per_capita'] = greenspace_sum['greenspace_area'] / greenspace_sum['pop21']
       
        # Compute per-capita (area / population) safely
        greenspace_sum['greenspace_per_capita'] = greenspace_sum['greenspace_area'] / greenspace_sum['pop21']
        # Areas with no greenspace -> 0
        greenspace_sum.loc[greenspace_sum['greenspace_area'] == 0, 'greenspace_per_capita'] = 0.0
        # Areas with no population (pop21 is NaN or <= 0) -> -1
        no_pop_mask = greenspace_sum['pop21'].isna() | (greenspace_sum['pop21'] <= 0)
        greenspace_sum.loc[no_pop_mask, 'greenspace_per_capita'] = -1.0

        # Save consolidated GeoJSON
        greenspace_sum = gpd.GeoDataFrame(greenspace_sum, geometry='geometry', crs='EPSG:32188')
        greenspace_sum = greenspace_sum.to_crs(epsg=4326)  # Convert back to WGS84 for GeoJSON
        greenspace_capita_path = os.path.join(output_dir, "greenspace_capita.geojson")
        greenspace_sum.to_file(greenspace_capita_path, driver="GeoJSON")
        print(f"Greenspace capita data saved to: {greenspace_capita_path}")

        # Remove unnecessary columns to reduce file size
        greenspace_final = greenspace_sum.drop(columns=[
            'CTNAME', 'PRUID', 'CDUID', 'CDNAME', 'CSDUID', 'CSDNAME',
            'CMAUID', 'CMANAME', 'CDTYPE', 'CSDTYPE', 'TYPE', 'pop21', 'LANDAREA'
        ], errors='ignore')  # Use `errors='ignore'` to avoid issues if columns are missing
        # Merge greenspace per capita value with census tract geometry


        #now we want to just display only the census geometry and per capita data
        # Ensure both GeoDataFrames use the same CRS
        census_gdf = gpd.read_file("data/censustracts.geojson")
        census_gdf = census_gdf.to_crs(epsg=4326)
        greenspace_sum = greenspace_sum.to_crs(epsg=4326)

        # Merge on CTUID to get geometry from census_gdf and greenspace_per_capita from greenspace_sum
        GS_capita_gdf = census_gdf[['CTUID', 'geometry']].merge(
            greenspace_sum[['CTUID', 'greenspace_per_capita']],
            on='CTUID',
            how='left'
        )

        # Save to GeoJSON
        GS_capita_gdf = gpd.GeoDataFrame(GS_capita_gdf, geometry='geometry', crs='EPSG:4326')
        
        greenspace_final_path = os.path.join(output_dir, "greenspace_capita.geojson")
        greenspace_final.to_file(greenspace_final_path, driver="GeoJSON")
        print(f"Final greenspace data saved to: {greenspace_final_path}")

    except Exception as e:
        print(f"An error occurred: {e}")

# Example usage
process_greenspace_data("raw-data/montreal/output_simplified_clipped_with_census.geojson", output_dir="data/montreal")


Greenspace capita data saved to: data/montreal\greenspace_capita.geojson
Final greenspace data saved to: data/montreal\greenspace_capita.geojson


now we want to just display only the census geometry and per capita data

In [ ]:
'''# Merge greenspace per capita value with census tract geometry
#  deprecated block since we made the function above
# Ensure both GeoDataFrames use the same CRS
census_gdf = census_gdf.to_crs(epsg=4326)
greenspace_sum = greenspace_sum.to_crs(epsg=4326)

# Merge on CTUID to get geometry from census_gdf and greenspace_per_capita from greenspace_sum
GS_capita_gdf = census_gdf[['CTUID', 'geometry']].merge(
    greenspace_sum[['CTUID', 'greenspace_per_capita']],
    on='CTUID',
    how='left'
)

# Save to GeoJSON
GS_capita_gdf = gpd.GeoDataFrame(GS_capita_gdf, geometry='geometry', crs='EPSG:4326')
GS_capita_gdf.to_file("data/montreal/greenspace_per_capita_only.geojson", driver="GeoJSON")
'''